This section is focused on testing different encodings, retrieval techniques, and weighting systems.

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import duckdb


In [ ]:
binary_train_result = pd.read_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/output_average.parquet", engine = "pyarrow")

In [2]:
signed_train_result = pd.read_parquet("/content/drive/MyDrive/Disease_engine_new_encoding/processed_data/output_average.parquet", engine = "pyarrow")

In [3]:
feature_scheme = pd.read_csv("/content/drive/MyDrive/Disease_engine_project/processed_data/features.csv")

Case 1:

```
  Encoding: Binary
  Retrieval: Cosine Similarity
```


In [5]:
def cosine_sim(record, space):
  correct = 0
  matrix_sqrts = []
  all_vectors = space.drop(columns="Diagnosis") ## Retrieve prifile vectors
  all_diagnosis = space[["Diagnosis"]].copy() ## Diagnose for each vector
  true_diag = record["Diagnosis"].to_list() ## Retrieve diagnosis of each patient
  predictions = []
  for j in range(0, record.shape[0]): ## For each patient record
    all_diagnosis = space[["Diagnosis"]].copy()
    diagnosis = record.iloc[j,0]
    vector = record.drop(columns="Diagnosis").iloc[j]
    scores = [] ## Array to store each diagnosis profile length to reduce repeated computations
    vector_sqrt = np.sqrt((vector**2).sum()) ## Calculate the length of the vector
    for i in range(0, space.shape[0]):
      avg_vector = all_vectors.iloc[i]
      numerator = (avg_vector * vector).sum()
      if j == 0: ## Store diagnosis profile vectors length for reducing repeated computations
        matrix_sqrts.append(np.sqrt((avg_vector**2).sum()))
      denominator = matrix_sqrts[i] * vector_sqrt
      cosine_score = numerator / denominator
      scores.append(cosine_score)
    all_diagnosis["Similarity_scores"] = np.array(scores) ## Add similarity score column to the diagnosis list
    all_diagnosis = all_diagnosis.sort_values(by="Similarity_scores", ascending = False) ## Sort
    predictions.append([all_diagnosis.iloc[0,0], all_diagnosis.iloc[1,0],
                        all_diagnosis.iloc[2,0], all_diagnosis.iloc[3,0], all_diagnosis.iloc[4,0]]) ## Retrieve top 5
  return true_diag, predictions

In [ ]:
true_diag = []
predictions = []
for i in range(10000, 130001, 10000):
  print(f"Reading parquet # {int(i/10000)}\n")
  validate_set = pd.read_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/validate_" + str(i) +".parquet", engine = "pyarrow")
  result = cosine_sim(validate_set, binary_train_result)
  true_diag = true_diag + result[0]
  predictions = predictions + result[1]

Reading parquet # 1

Reading parquet # 2

Reading parquet # 3

Reading parquet # 4

Reading parquet # 5

Reading parquet # 6

Reading parquet # 7

Reading parquet # 8

Reading parquet # 9

Reading parquet # 10

Reading parquet # 11

Reading parquet # 12

Reading parquet # 13



In [ ]:
real_data = pd.DataFrame({"Diagnosis": true_diag})
predicted_data = pd.DataFrame(predictions, columns=["Top_1", "Top_2", "Top_3", "Top_4", "Top_5"])

In [ ]:
predicted_data.head()
predicted_data["True_diagnosis"] = real_data

In [ ]:
predicted_data.to_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/validate_results.parquet", engine='pyarrow')

In [ ]:
binary_rep_results = pd.read_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/validate_results.parquet", engine='pyarrow')

In [ ]:
true_diagnosis = binary_rep_results["True_diagnosis"].to_list()
predicted_diagnosis = binary_rep_results["Top_1"].to_list()

diagnosis_list = binary_train_result.sort_values(by="Diagnosis", ascending=False)["Diagnosis"].to_list()

report = classification_report(true_diagnosis, predicted_diagnosis, labels=diagnosis_list)
print(report)

                                          precision    recall  f1-score   support

                          Whooping cough       1.00      1.00      1.00       534
                       Viral pharyngitis       1.00      1.00      1.00      8090
                         Unstable angina       1.00      0.87      0.93      2710
                                    URTI       1.00      1.00      1.00      8473
                            Tuberculosis       1.00      1.00      1.00      1977
                           Stable angina       0.86      1.00      0.93      2293
                Spontaneous rib fracture       1.00      1.00      1.00       768
                Spontaneous pneumothorax       1.00      1.00      1.00      1381
                Scombroid food poisoning       1.00      1.00      1.00      2203
                             Sarcoidosis       0.99      1.00      1.00      2972
                                     SLE       1.00      0.99      1.00      1555
               

From the results we can see that Unstable Angina and Acute Rhinosinusitis have a relatively low precision score while Stable Angina and Chronic Rhinosinusitis have a relatively low recall score.

In [ ]:
query_angina = "SELECT * FROM '/content/drive/MyDrive/Disease_engine_project/processed_data/output_average.parquet' WHERE DIAGNOSIS = 'Unstable angina' OR DIAGNOSIS = 'Stable angina'"
## query for retrieving the rows containing Unstable Angina and Stable Angina from the training results

In [ ]:
conn = duckdb.connect()

In [ ]:
low_score_vectors = conn.execute(query_angina).df()
## Query execution

In [ ]:
evidences = low_score_vectors.drop(columns="Diagnosis").columns.to_list()

In [ ]:
non_empty_angina = list(filter(lambda col:low_score_vectors[col].sum() != 0, evidences))
## Filter to obtain only evidences that contribute to differentiating between Unstable Angina and Stable Angina

In [ ]:
angina_important_df = low_score_vectors[ ["Diagnosis"] + non_empty_angina]

In [ ]:
angina_important_df.head()

,Diagnosis,E_55_V_29,E_55_V_30,E_55_V_31,E_55_V_33,E_55_V_39,E_55_V_55,E_55_V_56,E_55_V_101,E_55_V_127,...,E_105,E_104,E_79,E_71,E_69,E_70,E_204_V_10,E_204_V_4,E_143,E_225
0,Unstable angina,0.722463,0.146206,0.120034,0.094191,0.071973,0.519158,0.389192,0.677791,0.032197,...,0.817219,0.677886,0.710554,0.704434,0.762521,0.727170,0.969921,0.030079,0.655432,0.766287
1,Stable angina,0.687732,0.165107,0.134510,0.113680,0.088144,0.499264,0.405237,0.634187,0.046425,...,0.820124,0.676728,0.715328,0.704207,0.760812,0.727626,0.970462,0.029538,0.656723,0.768108


In [ ]:
abs(angina_important_df.drop(columns="Diagnosis").iloc[0] - angina_important_df.drop(columns="Diagnosis").iloc[1]).sort_values(ascending=False).head()
## Get the top 5 evidences that differentiate Unstable Angina and Stable Angina the most. As only the top 4 suppose a big difference, those are the ones that
## will be accounted for

,0
E_13,0.696291
E_14,0.658963
E_148,0.552580
E_50,0.527631
E_56_3,0.143984


In [ ]:
angina_important_df[["E_13", "E_14", "E_148", "E_50"]]

,E_13,E_14,E_148,E_50
0,0.696291,0.658963,0.55258,0.527631
1,0.000000,0.000000,0.00000,0.000000


As we can see from the results, the attributes that differentiate the most both diseases are E_13, E_14, E_148, and E_50.

In general terms, those evidences represent the worsening over time of symptoms such as chest pain, nausea, and increased sweating.
This results will set up the base for the development of a weighted cosine retireval system

The same analysis can be done for Chronic rhinosinusitis and Acute rhinosinusitis

In [ ]:
## Get data
query_rhino = "SELECT * FROM '/content/drive/MyDrive/Disease_engine_project/processed_data/output_average.parquet' WHERE DIAGNOSIS = 'Acute rhinosinusitis' OR DIAGNOSIS = 'Chronic rhinosinusitis'"
low_score_vectors = conn.execute(query_rhino).df()
evidences = low_score_vectors.drop(columns="Diagnosis").columns.to_list()

## Filter to obtain only columns that differentiate both Chronic and Acute Rhinosinusitis
non_empty_rhino = list(filter(lambda col:low_score_vectors[col].sum() != 0, evidences))
rhino_important_df = low_score_vectors[ ["Diagnosis"] + non_empty_rhino]

## Display evidences that differentiate both diagnosis the most
abs(rhino_important_df.drop(columns="Diagnosis").iloc[0] - rhino_important_df.drop(columns="Diagnosis").iloc[1]).sort_values(ascending=False).head()

,0
E_209,0.495728
E_91,0.354470
E_55_V_166,0.191457
E_181,0.190713
E_56_1,0.168910


As we can see, evidences E_209 and E_91 present the most difference between Chronic and Acute

Case 2:

```
  Encoding: Signed
  Retrieval: Cosine Similarity
```

The justification for a signed encoding is supporting even information storage and representation. The original training is executed signaling the precense of a symptom with the number 1 and the absence of such symptom with the number -1. Reserving the number 0 for the lack of answer/unknown answer.

The validation set is trained for a mmore realistic case when a patient affirms a series of symptoms while not giving an answer about other symptoms. This evaluation metric is meant to evaluate the decrease in performance of the model when, realistically speaking, a patient only affirms or rejects some of all the possible symptoms leaving a fraction of the symptoms with no answer.

In [6]:
true_diag = []
predictions = []
for i in range(10000, 130001, 10000):
  print(f"Reading parquet # {int(i/10000)}\n")
  validate_set = pd.read_parquet("/content/drive/MyDrive/Disease_engine_project/processed_data/validate_" + str(i) +".parquet", engine = "pyarrow")
  result = cosine_sim(validate_set, signed_train_result)
  true_diag = true_diag + result[0]
  predictions = predictions + result[1]

Reading parquet # 1

Reading parquet # 2

Reading parquet # 3

Reading parquet # 4

Reading parquet # 5

Reading parquet # 6

Reading parquet # 7

Reading parquet # 8

Reading parquet # 9

Reading parquet # 10

Reading parquet # 11

Reading parquet # 12

Reading parquet # 13



In [7]:
real_data = pd.DataFrame({"Diagnosis": true_diag})
predicted_data = pd.DataFrame(predictions, columns=["Top_1", "Top_2", "Top_3", "Top_4", "Top_5"])

In [8]:
predicted_data.head()
predicted_data["True_diagnosis"] = real_data

In [11]:
predicted_data.to_parquet("/content/drive/MyDrive/Disease_engine_new_encoding/processed_data/validate_results.parquet", engine='pyarrow')

In [12]:
signed_rep_results = pd.read_parquet("/content/drive/MyDrive/Disease_engine_new_encoding/processed_data/validate_results.parquet", engine='pyarrow')

In [16]:
true_diagnosis = signed_rep_results["True_diagnosis"].to_list()
predicted_diagnosis = signed_rep_results["Top_1"].to_list()

diagnosis_list = signed_train_result.sort_values(by="Diagnosis", ascending=False)["Diagnosis"].to_list()

report = classification_report(true_diagnosis, predicted_diagnosis, labels=diagnosis_list)
print(report)

                                          precision    recall  f1-score   support

                          Whooping cough       1.00      1.00      1.00       534
                       Viral pharyngitis       0.96      1.00      0.98      8090
                         Unstable angina       0.63      0.99      0.77      2710
                                    URTI       1.00      1.00      1.00      8473
                            Tuberculosis       1.00      1.00      1.00      1977
                           Stable angina       0.98      0.32      0.48      2293
                Spontaneous rib fracture       1.00      1.00      1.00       768
                Spontaneous pneumothorax       1.00      1.00      1.00      1381
                Scombroid food poisoning       1.00      1.00      1.00      2203
                             Sarcoidosis       0.97      1.00      0.98      2972
                                     SLE       1.00      0.99      0.99      1555
               